# V5c — DANN Re-evaluated with Alarm Aggregation

Faithful re-run of V5b DANN (`V5b_DANN_Tuned.ipynb`) — same architecture
(VAR(5) GC matrices → CNN feature extractor → label head + domain head with
gradient reversal, λ_max = 0.10), same hyperparameters, same V3 GC cache —
but the per-fold evaluation calls `evaluate_with_alarms` (K=5/M=12 vote,
30-min refractory) instead of the window-level evaluator.

AUC and AUC-PR remain threshold-free and identical to V5b. Only sensitivity,
specificity and FPR/h change.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports & config (mirrors V5b)
import os, sys, json, copy, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)

from config import (
    DATA_ROOT, CANONICAL_CHANNELS, N_CHANNELS, FS,
    STEP_SEC, EXCLUDED_PATIENTS, GC_MATRICES_DIR, RESULTS_DIR,
    INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS, RANDOM_SEED,
    BATCH_SIZE, ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from metrics import evaluate_with_alarms
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# V5b hyperparameters
V5b_EPOCHS    = 30
V5b_PATIENCE  = 8
V5b_LR        = 1e-3
V5b_LAMBDA_MAX= 0.10
V5b_DROPOUT   = 0.5
V5b_VAL_FRAC  = 0.15
V5b_MAX_WIN_PER_PAT = 1500

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  λ_max={V5b_LAMBDA_MAX}')


Device: cpu  λ_max=0.1


In [2]:
# Cell 1 — Load V3 GC cache (mirrors V5b loader)
cache_root = Path(GC_MATRICES_DIR)
patients_all = sorted([
    p for p in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, p))
    and p.startswith('chb') and p not in EXCLUDED_PATIENTS
])
patient_data = {}
for pid in patients_all:
    pdir = cache_root / pid
    if not pdir.exists(): continue
    gc_files = sorted(pdir.glob('*_gc.npy'))
    if not gc_files: continue
    gc_list, lb_list = [], []
    for gp in gc_files:
        lp = gp.with_name(gp.name.replace('_gc.npy', '_labels.npy'))
        if lp.exists():
            gc_list.append(np.load(gp)); lb_list.append(np.load(lp))
    if not gc_list: continue
    X = np.concatenate(gc_list, axis=0).astype(np.float32)
    y = np.concatenate(lb_list, axis=0).astype(np.int8)
    n_pre, n_int = int((y==1).sum()), int((y==0).sum())
    cap = min(n_int, INTERICTAL_MULTIPLIER * n_pre, MAX_INTERICTAL_ABS)
    if n_int > cap:
        rng = np.random.default_rng(RANDOM_SEED + hash(pid) % 10_000)
        int_idx = np.where(y == 0)[0]; pre_idx = np.where(y == 1)[0]
        keep = np.sort(np.concatenate([pre_idx, rng.choice(int_idx, size=cap, replace=False)]))
        X, y = X[keep], y[keep]
    if n_pre == 0: continue
    patient_data[pid] = (X, y)

patient_ids = sorted(patient_data.keys())
print(f'Loaded {len(patient_ids)} patients')


Loaded 21 patients


In [3]:
# Cell 2 — DANN architecture (replicated from V5b)
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = float(lambda_); return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

def grad_reverse(x, lambda_=1.0):
    return GradReverse.apply(x, lambda_)


class DANN(nn.Module):
    def __init__(self, n_patients, n_channels=N_CHANNELS, dropout=V5b_DROPOUT):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        with torch.no_grad():
            self.flat_size = self.features(torch.zeros(1,1,n_channels,n_channels)).numel()
        self.label_clf = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(self.flat_size, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, 1), nn.Sigmoid())
        self.domain_clf = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flat_size, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(128, n_patients))

    def forward(self, x, lambda_=1.0):
        f = self.features(x)
        return self.label_clf(f), self.domain_clf(grad_reverse(f, lambda_))


class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.gamma = gamma; self.pos_weight = pos_weight
    def forward(self, p, t):
        p = p.squeeze(1).clamp(1e-7, 1 - 1e-7); t = t.float()
        alpha = torch.where(t == 1, torch.full_like(t, self.pos_weight), torch.ones_like(t))
        bce = -(t * torch.log(p) + (1 - t) * torch.log(1 - p))
        pt  = torch.where(t == 1, p, 1 - p)
        return (alpha * (1 - pt) ** self.gamma * bce).mean()


class DANNDataset(Dataset):
    def __init__(self, X, y, d):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.d = torch.tensor(d, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i], self.d[i]


def lambda_schedule(epoch, max_epochs, lam_max=V5b_LAMBDA_MAX):
    p = epoch / max(max_epochs - 1, 1)
    return (2.0 / (1.0 + np.exp(-10 * p)) - 1.0) * lam_max

print('DANN architecture loaded.')


DANN architecture loaded.


In [4]:
# Cell 3 — Training + prediction
def train_dann(X_tr, y_tr, d_tr, X_va, y_va, n_patients, verbose=False):
    pos_w = ((d_tr.shape[0] - (y_tr == 1).sum()) / max((y_tr == 1).sum(), 1))
    model = DANN(n_patients=n_patients).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=V5b_LR, weight_decay=1e-4)
    focal = BinaryFocalLoss(gamma=2.0, pos_weight=float(pos_w))
    ce    = nn.CrossEntropyLoss()
    ds_tr = DANNDataset(X_tr, y_tr, d_tr)
    ld_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    best_va = float('inf'); best_state = None; patience_ct = 0
    for ep in range(V5b_EPOCHS):
        lam = lambda_schedule(ep, V5b_EPOCHS)
        model.train()
        for xb, yb, db in ld_tr:
            xb, yb, db = xb.to(DEVICE), yb.to(DEVICE), db.to(DEVICE)
            p_lbl, p_dom = model(xb, lambda_=lam)
            loss = focal(p_lbl, yb) + ce(p_dom, db)
            opt.zero_grad(); loss.backward(); opt.step()
        # validation
        model.eval()
        with torch.no_grad():
            xv = torch.tensor(X_va, dtype=torch.float32).unsqueeze(1).to(DEVICE)
            yv = torch.tensor(y_va, dtype=torch.float32).to(DEVICE)
            pv, _ = model(xv, lambda_=0.0)
            v_loss = focal(pv, yv).item()
        if v_loss < best_va:
            best_va = v_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_ct = 0
        else:
            patience_ct += 1
            if patience_ct >= V5b_PATIENCE: break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_proba(model, X):
    model.eval()
    x = torch.tensor(X, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    p, _ = model(x, lambda_=0.0)
    return p.squeeze(1).cpu().numpy()

print('Trainer ready.')


Trainer ready.


In [5]:
# Cell 4 — LOPO loop with smoothing + alarm-level operational threshold
# Post-processing follows Truong 2018 / Bandarabadi 2015 / Mormann 2007:
#   raw probs → moving-average smoothing
#             → operational threshold (Sens maximised s.t. ALARM-level FPR/h ≤ 1.0)
#             → K-of-M sliding-window vote → 30-min refractory suppression.
from alarm_postproc import smooth_probs, operational_threshold

SMOOTH_WINDOW = 10           # 10 windows ≈ 100 s (matches M=12 voting window)
TARGET_FPR_H  = 1.0          # Mormann 2007 clinical threshold (alarm-level)

rows_alarm, rows_cmp = [], []
t_total = time.time()
n_patients = len(patient_ids)
pid_to_idx = {p:i for i,p in enumerate(patient_ids)}

for fi, test_pid in enumerate(patient_ids, 1):
    train_pids = [p for p in patient_ids if p != test_pid]
    Xs, ys, ds = [], [], []
    for tp in train_pids:
        Xp, yp = patient_data[tp]
        n = len(yp)
        if n > V5b_MAX_WIN_PER_PAT:
            rng = np.random.default_rng(RANDOM_SEED + fi + hash(tp) % 100)
            idx = rng.choice(n, V5b_MAX_WIN_PER_PAT, replace=False)
            Xp, yp = Xp[idx], yp[idx]
        Xs.append(Xp); ys.append(yp)
        ds.append(np.full(len(yp), pid_to_idx[tp]))
    X_all = np.concatenate(Xs); y_all = np.concatenate(ys); d_all = np.concatenate(ds)

    scaler = StandardScaler()
    flat = X_all.reshape(len(X_all), -1)
    flat = scaler.fit_transform(flat)
    X_all = flat.reshape(X_all.shape)

    tr_idx, va_idx = train_test_split(np.arange(len(y_all)), test_size=V5b_VAL_FRAC,
                                      random_state=RANDOM_SEED, stratify=y_all)
    X_tr, y_tr, d_tr = X_all[tr_idx], y_all[tr_idx], d_all[tr_idx]
    X_va, y_va = X_all[va_idx], y_all[va_idx]

    X_te, y_te = patient_data[test_pid]
    X_te = scaler.transform(X_te.reshape(len(X_te), -1)).reshape(X_te.shape)

    print(f'[Fold {fi:2d}/{n_patients}] {test_pid}  train={len(y_tr)}  val={len(y_va)}  test={len(y_te)}')
    model = train_dann(X_tr, y_tr, d_tr, X_va, y_va, n_patients)
    probs_raw = predict_proba(model, X_te)

    if len(np.unique(y_te)) < 2: continue

    # --- post-processing pipeline ---
    probs    = smooth_probs(probs_raw, window=SMOOTH_WINDOW)
    threshold = operational_threshold(
        probs, y_te, STEP_SEC, target_fpr_h=TARGET_FPR_H,
        K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
    )

    a = evaluate_with_alarms(
        y_true=y_te, probs=probs, threshold=threshold,
        K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
        patient_id=test_pid,
    )
    a['model'] = 'DANN'; a['threshold'] = threshold
    rows_alarm.append(a)

    # window-level reference (raw probs, raw Youden-J)
    fpr_a, tpr_a, thr = roc_curve(y_te, probs_raw)
    thr_raw = float(np.clip(thr[np.argmax(tpr_a - fpr_a)], 0.05, 0.95))
    pred_w = (probs_raw >= thr_raw).astype(int)
    tp=int(((pred_w==1)&(y_te==1)).sum()); fp=int(((pred_w==1)&(y_te==0)).sum())
    tn=int(((pred_w==0)&(y_te==0)).sum()); fn=int(((pred_w==0)&(y_te==1)).sum())
    sens_w = tp/max(tp+fn,1)
    hours_int = (y_te==0).sum() * STEP_SEC / 3600.0
    fpr_w = fp/max(hours_int, 1e-9)
    rows_cmp.append(dict(patient=test_pid, model='DANN',
        auc=a['auc'], auc_pr=a['auc_pr'], threshold=threshold,
        sens_window=sens_w, fpr_h_window=fpr_w,
        sens_alarm=a['sensitivity'], fpr_h_alarm=a['fpr_per_hour']))
    print(f'  AUC={a["auc"]:.3f}  AUC-PR={a["auc_pr"]:.3f}  '
          f'thr={threshold:.2f}  Sens(a)={a["sensitivity"]:.3f}  '
          f'FPR/h(w)={fpr_w:.1f} → FPR/h(a)={a["fpr_per_hour"]:.2f}')

print(f'\n  Total time: {(time.time()-t_total)/60:.1f}min')


[Fold  1/21] chb01  train=25500  val=4500  test=1776
  AUC=0.313  AUC-PR=0.116  thr=0.55  Sens(a)=0.000  FPR/h(w)=355.6 → FPR/h(a)=0.97
[Fold  2/21] chb02  train=25500  val=4500  test=1776
  AUC=0.390  AUC-PR=0.179  thr=0.43  Sens(a)=0.003  FPR/h(w)=13.1 → FPR/h(a)=0.97
[Fold  3/21] chb03  train=25500  val=4500  test=2664
  AUC=0.416  AUC-PR=0.292  thr=0.47  Sens(a)=0.002  FPR/h(w)=21.2 → FPR/h(a)=0.65
[Fold  4/21] chb04  train=25500  val=4500  test=1776
  AUC=0.401  AUC-PR=0.159  thr=0.53  Sens(a)=0.003  FPR/h(w)=55.2 → FPR/h(a)=0.97
[Fold  5/21] chb05  train=25500  val=4500  test=2664
  AUC=0.476  AUC-PR=0.179  thr=0.48  Sens(a)=0.004  FPR/h(w)=138.0 → FPR/h(a)=0.97
[Fold  6/21] chb06  train=25500  val=4500  test=6037
  AUC=0.606  AUC-PR=0.303  thr=0.46  Sens(a)=0.004  FPR/h(w)=170.1 → FPR/h(a)=0.94
[Fold  7/21] chb07  train=25500  val=4500  test=2670
  AUC=0.705  AUC-PR=0.355  thr=0.43  Sens(a)=0.007  FPR/h(w)=186.2 → FPR/h(a)=0.97
[Fold  8/21] chb08  train=25500  val=4500  test=444

In [6]:
# Cell 5 — Save and summarise
pd.DataFrame(rows_alarm).to_csv(Path(RESULTS_DIR) / 'lopo_v5c_alarm_DANN.csv', index=False)
pd.DataFrame(rows_cmp).to_csv(Path(RESULTS_DIR) / 'lopo_v5c_compare_DANN.csv', index=False)
df = pd.DataFrame(rows_cmp)
print(f'{"Model":<6} {"AUC":>6} {"AUC-PR":>7} '
      f'{"Sens(w)":>8} {"FPR/h(w)":>10} {"Sens(a)":>8} {"FPR/h(a)":>10}')
print(f'{"DANN":<6} {df.auc.mean():>6.3f} {df.auc_pr.mean():>7.3f} '
      f'{df.sens_window.mean():>8.3f} {df.fpr_h_window.mean():>10.1f} '
      f'{df.sens_alarm.mean():>8.3f} {df.fpr_h_alarm.mean():>10.2f}')
print('\nOutputs: lopo_v5c_alarm_DANN.csv, lopo_v5c_compare_DANN.csv')


Model     AUC  AUC-PR  Sens(w)   FPR/h(w)  Sens(a)   FPR/h(a)
DANN    0.524   0.216    0.599      177.0    0.003       0.89

Outputs: lopo_v5c_alarm_DANN.csv, lopo_v5c_compare_DANN.csv
